<a href="https://colab.research.google.com/github/PatrickCalorioCarvalho/CA015ICMineracaoDeDados202601/blob/main/Pre_processamento_P1_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from category_encoders import BinaryEncoder, CountEncoder

## Pequena ilustração didática das quatro técnicas

Considere uma variável categórica simples chamada `cor`.

In [3]:
exemplo = pd.DataFrame({"cor":["Azul","Verde","Vermelho", "Azul", "Verde", "Azul"],
                        "nome":["Danilo","Joao","Danilo","Pedro", "Maria", "Ana"]})

In [4]:
exemplo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cor     6 non-null      object
 1   nome    6 non-null      object
dtypes: object(2)
memory usage: 228.0+ bytes


### LabelEncoder

O **LabelEncoder** troca cada categoria por um número inteiro.

Exemplo:

| cor | cor_label |
|---|---:|
| Azul | 0 |
| Verde | 1 |
| Vermelho | 2 |

**Atenção:** o modelo pode interpretar que existe uma ordem artificial entre as categorias, por exemplo, `Vermelho > Verde > Azul`, mesmo que isso não faça sentido.

In [5]:
ex_label = exemplo.copy()
le = LabelEncoder()
ex_label["cor_LE"] = le.fit_transform(ex_label['cor'])

In [6]:
ex_label

,cor,nome,cor_LE
0,Azul,Danilo,0
1,Verde,Joao,1
2,Vermelho,Danilo,2
3,Azul,Pedro,0
4,Verde,Maria,1
5,Azul,Ana,0


### OneHotEncoder

O **OneHotEncoder** cria uma coluna binária para cada categoria.

Exemplo:

| cor | cor_Azul | cor_Verde | cor_Vermelho |
|---|---:|---:|---:|
| Azul | 1 | 0 | 0 |
| Verde | 0 | 1 | 0 |
| Vermelho | 0 | 0 | 1 |

Essa técnica evita a ideia de ordem artificial, mas pode aumentar muito a quantidade de colunas quando há muitas categorias.

In [7]:
ohe_copy = exemplo.copy()
ohe = OneHotEncoder(sparse_output=False)
array_ohe = ohe.fit_transform(ohe_copy[['cor']]).astype(int)
cols_ohe = ohe.get_feature_names_out(['cor'])
ex_ohe = pd.DataFrame(array_ohe, columns=cols_ohe)
pd.concat([ohe_copy, ex_ohe], axis = 1)

,cor,nome,cor_Azul,cor_Verde,cor_Vermelho
0,Azul,Danilo,1,0,0
1,Verde,Joao,0,1,0
2,Vermelho,Danilo,0,0,1
3,Azul,Pedro,1,0,0
4,Verde,Maria,0,1,0
5,Azul,Ana,1,0,0


### BinaryEncoder com `category_encoders`

O **BinaryEncoder** transforma as categorias em códigos binários. Ele costuma gerar menos colunas que o OneHotEncoder quando há muitas categorias.

Ideia simplificada:

| cor | código inteiro | binário |
|---|---:|---:|
| Azul | 1 | 01 |
| Verde | 2 | 10 |
| Vermelho | 3 | 11 |

É útil quando a variável categórica possui muitas categorias, reduzindo a dimensionalidade em comparação ao One-Hot Encoding.

In [8]:
be = BinaryEncoder(cols=['cor'])
ex_binary = be.fit_transform(exemplo['cor'])
ex_binary
# pd.concat([exemplo['cor'],ex_binary], axis=1)

,cor_0,cor_1
0,0,1
1,1,0
2,1,1
3,0,1
4,1,0
5,0,1


### CountEncoder / Frequency Encoding com `category_encoders`

O **CountEncoder** substitui cada categoria pela quantidade de vezes que ela aparece na base.

Exemplo:

| cor | frequência |
|---|---:|
| Azul | 3 |
| Verde | 2 |
| Vermelho | 1 |

Essa técnica pode ser útil quando a frequência de uma categoria carrega informação relevante. Porém, categorias diferentes com a mesma frequência receberão o mesmo valor.

In [9]:
count_encoder = CountEncoder(cols=['cor'])
ex_count = count_encoder.fit_transform(exemplo['cor'])
# ex_count
pd.concat([exemplo['cor'],ex_count], axis= 1)

,cor,cor
0,Azul,3
1,Verde,2
2,Vermelho,1
3,Azul,3
4,Verde,2
5,Azul,3


Estudo de caso - Stroke Prediction

Acesso: https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

In [11]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [13]:
cols_cat = df.select_dtypes(include=["object"]).columns.tolist()

In [14]:
cols_cat

['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

In [15]:
for cat in cols_cat:
  display(df[cat].unique())

array(['Male', 'Female', 'Other'], dtype=object)

array(['Yes', 'No'], dtype=object)

array(['Private', 'Self-employed', 'Govt_job', 'children', 'Never_worked'],
      dtype=object)

array(['Urban', 'Rural'], dtype=object)

array(['formerly smoked', 'never smoked', 'smokes', 'Unknown'],
      dtype=object)

In [16]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
